# 01 — Documents

**LLMEvalIQ v1 — Phases 2 & 3**

Phase 2 picks the corpus and proves it is usable.
Phase 3 turns it into the gold evaluation set (`data/evaluation/questions.csv`).

Ground truth is the foundation of everything downstream: Recall@K and
correctness are both meaningless without it.

## Setup

In [ ]:
# Run this first in every notebook.
# Notebooks live in notebooks/, but our code lives in src/ — this makes
# `from src.ingestion import ...` work by putting the project root on the path.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

DOCUMENTS_DIR = PROJECT_ROOT / "data" / "documents"
EVALUATION_DIR = PROJECT_ROOT / "data" / "evaluation"

print("Project root:", PROJECT_ROOT)
print("Documents   :", DOCUMENTS_DIR)

## Phase 2.1 — Inspect the corpus

Never trust a folder of PDFs you have not measured. Run this **before**
writing a single evaluation question.

In [ ]:
"""Inspect every PDF in data/documents/ before trusting any of it.

Why this cell exists: the single most common way a RAG project dies is a
corpus of scanned PDFs. They look identical to real PDFs in Finder, but they
are images — pypdf extracts ~0 characters and every downstream metric is
garbage. `chars_per_page` catches that in 5 seconds instead of 5 hours.
"""
import pandas as pd
from pypdf import PdfReader

rows = []
for pdf_path in sorted(DOCUMENTS_DIR.glob("*.pdf")):
    reader = PdfReader(pdf_path)
    n_pages = len(reader.pages)
    # `or ""` guards against pages that yield None (images, blank pages).
    text = "\n".join(page.extract_text() or "" for page in reader.pages)
    rows.append({
        "document": pdf_path.name,
        "pages": n_pages,
        "characters": len(text),
        "chars_per_page": round(len(text) / n_pages) if n_pages else 0,
    })

corpus = pd.DataFrame(rows)
corpus

### Interpreting `chars_per_page`

| Value | Meaning |
|---|---|
| 1,500 - 3,500 | Healthy text PDF ✅ |
| 200 - 1,000 | Sparse — heavy tables/figures. Usable, watch it. |
| < 200 | **Scanned image.** pypdf cannot read it. Replace or OCR. ❌ |

A scanned PDF is the single most common way this project fails silently:
retrieval returns nothing useful and every metric reads as a model problem
when it is really a data problem.

## Phase 2.2 — Health check against the targets

In [ ]:
"""Three checks. Fix any failure now — not after you've written 100 questions."""
from src.ingestion import CHUNK_SIZE, CHUNK_OVERLAP

if corpus.empty:
    raise SystemExit("No PDFs found. Put them in data/documents/ first.")

n_docs = len(corpus)
total_pages = int(corpus["pages"].sum())
total_chars = int(corpus["characters"].sum())

# A text-based PDF yields roughly 1,500-3,500 chars/page. Anything under ~200
# is almost certainly a scan, and needs OCR or replacing.
suspect = corpus[corpus["chars_per_page"] < 200]

def check(label: str, ok: bool, detail: str) -> None:
    print(f"{'PASS' if ok else 'FAIL'}  {label:<22} {detail}")

print("CORPUS HEALTH CHECK")
print("-" * 62)
check("Document count", 10 <= n_docs <= 20, f"{n_docs} (target 10-20)")
check("Total pages", 200 <= total_pages <= 500, f"{total_pages} (target 200-500)")
check("Text extraction", suspect.empty,
      "all readable" if suspect.empty else f"{len(suspect)} suspect: {list(suspect['document'])}")
print("-" * 62)

# Each chunk advances by (CHUNK_SIZE - CHUNK_OVERLAP) characters, because the
# overlap is re-read by the next chunk. That stride is what sets the count.
stride = CHUNK_SIZE - CHUNK_OVERLAP
print(f"Total characters      : {total_chars:,}")
print(f"Estimated chunks      : ~{total_chars // stride:,}  (stride {stride} chars)")
print(f"One-time embed cost   : well under $0.10 at these volumes")

## Phase 2.3 — Write the manifest

The manifest is your corpus documentation. Its `document` column is the exact
string `questions.csv` will use in `relevant_document` — the join key that
Recall@K scores against.

In [ ]:
"""Write a manifest: what each document is, so questions.csv can reference it.

The `document` column here becomes the `relevant_document` column in
questions.csv. They must match EXACTLY — that string is the join key the
Recall@K metric depends on.
"""
manifest_path = DOCUMENTS_DIR / "MANIFEST.csv"
manifest = corpus[["document", "pages"]].copy()
manifest["title"] = ""        # e.g. "Siemens Annual Report 2024"
manifest["entity"] = ""       # e.g. "Siemens AG"
manifest["fiscal_year"] = ""  # e.g. 2024
manifest["source_url"] = ""   # where you downloaded it — for reproducibility

manifest.to_csv(manifest_path, index=False)
print(f"Wrote {manifest_path}")
print("Now open it and fill in the blank columns by hand.")
manifest

---

# Phase 3 — Gold Evaluation Set

Everything downstream is scored against `data/evaluation/questions.csv`.
It is the ruler. If the ruler is wrong, every measurement is wrong — and
nothing in the pipeline will tell you.

**Schema**

| Column | Meaning |
|---|---|
| `question_id` | Stable unique key, e.g. `q001`. Never reuse or renumber. |
| `question` | What a real user would ask, in their words. |
| `reference_answer` | The correct answer, written by you from the document. |
| `relevant_document` | Exact PDF filename containing the answer. **Join key.** |
| `difficulty` | `easy` / `medium` / `difficult` |

**Target mix:** 60 easy, 30 medium, 10 difficult.

## Phase 3.1 — Validate the gold set

Run this after every edit. It is cheap; a wrong ruler is not.

In [ ]:
"""Validate the gold evaluation set.

Run this EVERY time you edit questions.csv. A broken gold set does not raise
an error during evaluation — it silently produces wrong scores. This cell is
the only thing standing between you and a month of misleading numbers.
"""
import pandas as pd

QUESTIONS_PATH = EVALUATION_DIR / "questions.csv"
REQUIRED_COLUMNS = [
    "question_id", "question", "reference_answer", "relevant_document", "difficulty",
]
TARGET_MIX = {"easy": 60, "medium": 30, "difficult": 10}

questions = pd.read_csv(QUESTIONS_PATH, dtype=str).fillna("")
problems: list[str] = []

# --- 1. Schema -------------------------------------------------------------
missing_cols = [c for c in REQUIRED_COLUMNS if c not in questions.columns]
if missing_cols:
    problems.append(f"Missing columns: {missing_cols}")

# --- 2. No blanks ----------------------------------------------------------
for col in REQUIRED_COLUMNS:
    if col in questions.columns:
        blanks = questions.index[questions[col].str.strip() == ""].tolist()
        if blanks:
            problems.append(f"Blank '{col}' at row(s) {blanks[:5]}")

# --- 3. question_id is a usable primary key --------------------------------
dupe_ids = questions["question_id"][questions["question_id"].duplicated()].tolist()
if dupe_ids:
    problems.append(f"Duplicate question_id: {dupe_ids[:5]}")

# --- 4. Difficulty vocabulary ----------------------------------------------
bad_difficulty = sorted(set(questions["difficulty"]) - set(TARGET_MIX))
if bad_difficulty:
    problems.append(f"Unknown difficulty values: {bad_difficulty} (use {list(TARGET_MIX)})")

# --- 5. Duplicate questions ------------------------------------------------
norm = questions["question"].str.lower().str.strip()
dupe_q = questions["question"][norm.duplicated()].tolist()
if dupe_q:
    problems.append(f"Duplicate question text: {dupe_q[:3]}")

# --- 6. JOIN INTEGRITY — the one that silently ruins Recall@K --------------
# relevant_document must match a real filename in data/documents/ EXACTLY.
# A typo here is scored as a retrieval miss forever, and looks like a model bug.
on_disk = {p.name for p in DOCUMENTS_DIR.glob("*.pdf")}
referenced = set(questions["relevant_document"]) - {""}
orphans = sorted(referenced - on_disk)
if orphans:
    problems.append(f"relevant_document not found on disk: {orphans[:5]}")

unused = sorted(on_disk - referenced)

# --- Report ----------------------------------------------------------------
print(f"GOLD SET VALIDATION — {len(questions)} questions")
print("-" * 62)
for level, target in TARGET_MIX.items():
    actual = int((questions["difficulty"] == level).sum())
    flag = "OK " if actual == target else "-->"
    print(f"{flag} {level:<10} {actual:>3} / {target}")
print("-" * 62)

if problems:
    print(f"{len(problems)} PROBLEM(S) — fix before running any evaluation:\n")
    for p in problems:
        print(f"  x {p}")
else:
    print("No problems found.")

if unused:
    print(f"\nNote: {len(unused)} document(s) have no questions at all:")
    for name in unused[:5]:
        print(f"  - {name}")
    print("  Documents with zero questions are dead weight in the index —")
    print("  they can only ever hurt retrieval precision, never help recall.")

## Phase 3.2 — Coverage

Check that questions are spread across the corpus rather than clustered in
one or two documents.

In [ ]:
"""How are questions spread across documents and difficulty?

Concentration is a silent bias. If 40 of your 100 questions come from one
report, your headline scores mostly describe how well the pipeline handles
THAT report — not your corpus.
"""
if not questions.empty:
    coverage = (
        questions.pivot_table(
            index="relevant_document",
            columns="difficulty",
            values="question_id",
            aggfunc="count",
            fill_value=0,
        )
        .reindex(columns=["easy", "medium", "difficult"], fill_value=0)
    )
    coverage["total"] = coverage.sum(axis=1)
    coverage = coverage.sort_values("total", ascending=False)

    display(coverage)

    share = coverage["total"] / coverage["total"].sum()
    print(f"\nMost-represented document: {share.max():.0%} of all questions")
    if share.max() > 0.25:
        print("WARNING: over 25% from one document — spread the questions wider.")
else:
    print("questions.csv is empty — nothing to plot yet.")